In [112]:
!pip install spacy
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 50.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 44.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 51.5 MB/s eta 0:00:00


In [96]:
import os
import re
import contractions

In [97]:
def extract_subtitles_with_timestamps(ass_file, output_txt):
    with open(ass_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    subtitles = []
    in_events = False

    for line in lines:
        # Detecta el inicio de la sección [Events]
        if line.strip().lower() == "[events]":
            in_events = True
            continue

        if in_events:
            # Extrae solo las líneas de subtítulos (que empiezan con "Dialogue:")
            if line.startswith("Dialogue:"):
                # Divide la línea en columnas usando la coma como separador
                parts = line.split(",", 9)  # Separa en máximo 10 partes
                if len(parts) > 9:
                    author = parts[4].strip()    # Tiempo de fin
                    subtitle_text = parts[9].strip()  # El texto del subtítulo
                    # Formatea la salida incluyendo las marcas de tiempo
                    subtitles.append(f"{author},{subtitle_text}")

    # Guarda los subtítulos en un archivo .txt
    with open(output_txt, 'w', encoding='utf-8') as out_file:
        out_file.write("\n".join(subtitles))


In [98]:
def process_folder(folder_path, output_folder):
    # Crea la carpeta de salida si no existe
    os.makedirs(output_folder, exist_ok=True)

    # Itera sobre todos los archivos .ass en el folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".ass"):
            ass_path = os.path.join(folder_path, file_name)
            txt_path = os.path.join(output_folder, file_name.replace(".ass", ".txt"))

            print(f"Procesando: {file_name} -> {txt_path}")
            extract_subtitles_with_timestamps(ass_path, txt_path)

In [99]:
output_folder = "OUT_JujutsuKaisen01"
process_folder("JujutsuKaisen01", output_folder)

Procesando: [Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].3.eng.ass -> OUT_JujutsuKaisen01/[Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].3.eng.txt
Procesando: [Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].9.spa.ass -> OUT_JujutsuKaisen01/[Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].9.spa.txt


In [100]:
def limpiar_texto(texto):
    """Elimina signos de puntuación y caracteres no alfabéticos."""
    # texto = re.sub(r'[^a-zA-Z\s¿?¡!]', '', texto)
    texto = re.sub(r'\{.*?\}', '', texto)
    return texto.lower().strip()

In [101]:
def expand_contractions(text):
    return contractions.fix(text)

In [102]:
# Realizar limpieza de los archivos ya unificados

os.makedirs(output_folder, exist_ok=True)

output_folder_final = "CLEAN_OUT_JujutsuKaisen01"
os.makedirs(output_folder_final, exist_ok=True)

for file_name in os.listdir(output_folder):
    with open(output_folder + "/" + file_name, 'r', encoding='utf-8') as file:
        content = file.read().replace("\\N", " ")
        content_expandido = expand_contractions(content)
        texto_limpio = limpiar_texto(content_expandido)
        file.close()

        with open(output_folder_final + "/" + file_name, 'w', encoding='utf-8') as file:
            file.write(texto_limpio)    

        

In [107]:
def group_subtitles_by_author_to_files(input_file, output_dir):
    """
    Lee un archivo con formato "autor,texto", agrupa las líneas consecutivas del mismo autor
    y guarda, para cada autor, un archivo independiente en 'output_dir' con los grupos concatenados.
    """
    # Diccionario para almacenar grupos por autor.
    # Cada clave será un autor y su valor una lista de grupos (secuencias consecutivas).
    grouped = {}

    current_author = None
    current_text = ""

    # Lee el archivo de entrada
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Recorre línea a línea
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Separa la línea en dos partes: autor y texto (máximo 1 división)
        parts = line.split(",", 1)
        if len(parts) != 2:
            continue  # Si la línea no cumple el formato, la omite

        author, text = parts[0].strip(), parts[1].strip()

        # Si es la primera línea o si el autor es el mismo que el anterior, se concatena el texto
        if current_author is None or author == current_author:
            if current_author is None:
                current_author = author
            current_text += (" " if current_text else "") + text
        else:
            # Cuando el autor cambia, se guarda el grupo anterior en el diccionario
            if current_author not in grouped:
                grouped[current_author] = []
            grouped[current_author].append(current_text)

            # Se reinicia el grupo para el nuevo autor
            current_author = author
            current_text = text

    # Guarda el último grupo
    if current_author is not None:
        if current_author not in grouped:
            grouped[current_author] = []
        grouped[current_author].append(current_text)

    # Asegúrate de que el directorio de salida exista
    os.makedirs(output_dir, exist_ok=True)

    # Escribe un archivo para cada autor con los grupos concatenados (separados por una línea en blanco)
    for author, groups in grouped.items():
        # Genera un nombre de archivo simple (puedes mejorar la sanitización si es necesario)
        filename = os.path.join(output_dir, f"{author if author else 'sin_autor'}.txt")
        with open(filename, 'w', encoding='utf-8') as out_file:
            out_file.write("\n\n".join(groups))
        print(f"Archivo creado para '{author}': {filename}")

In [110]:
input_file = "/workspaces/jupyter/NLP/proyecto_final/CLEAN_OUT_JujutsuKaisen01/[Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].9.spa.txt"   # Archivo de subtítulos en inglés con formato: autor,texto
output_dir = "subtitles_by_author_es"
group_subtitles_by_author_to_files(input_file, output_dir)

Archivo creado para 'gojou': subtitles_by_author_es/gojou.txt
Archivo creado para 'yuuji': subtitles_by_author_es/yuuji.txt
Archivo creado para 'logo': subtitles_by_author_es/logo.txt
Archivo creado para 'episodio': subtitles_by_author_es/episodio.txt
Archivo creado para 'cartel': subtitles_by_author_es/cartel.txt
Archivo creado para 'enfermera': subtitles_by_author_es/enfermera.txt
Archivo creado para 'grandpa': subtitles_by_author_es/grandpa.txt
Archivo creado para 'fushiguro': subtitles_by_author_es/fushiguro.txt
Archivo creado para 'sign': subtitles_by_author_es/sign.txt
Archivo creado para 'all&screen': subtitles_by_author_es/all&screen.txt
Archivo creado para 'presidente': subtitles_by_author_es/presidente.txt
Archivo creado para 'sasaki': subtitles_by_author_es/sasaki.txt
Archivo creado para '': subtitles_by_author_es/sin_autor.txt
Archivo creado para 'takagi': subtitles_by_author_es/takagi.txt
Archivo creado para 'alumno': subtitles_by_author_es/alumno.txt
Archivo creado para '

In [111]:
input_file = "/workspaces/jupyter/NLP/proyecto_final/CLEAN_OUT_JujutsuKaisen01/[Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].3.eng.txt"   # Archivo de subtítulos en inglés con formato: autor,texto
output_dir = "subtitles_by_author_eng"
group_subtitles_by_author_to_files(input_file, output_dir)

Archivo creado para 'gojou': subtitles_by_author_eng/gojou.txt
Archivo creado para 'yuuji': subtitles_by_author_eng/yuuji.txt
Archivo creado para '': subtitles_by_author_eng/sin_autor.txt
Archivo creado para 'screen': subtitles_by_author_eng/screen.txt
Archivo creado para 'plate': subtitles_by_author_eng/plate.txt
Archivo creado para 'nurse': subtitles_by_author_eng/nurse.txt
Archivo creado para 'grandpa': subtitles_by_author_eng/grandpa.txt
Archivo creado para 'sign': subtitles_by_author_eng/sign.txt
Archivo creado para 'fushiguro': subtitles_by_author_eng/fushiguro.txt
Archivo creado para 'all&screen': subtitles_by_author_eng/all&screen.txt
Archivo creado para 'sidebar': subtitles_by_author_eng/sidebar.txt
Archivo creado para 'president': subtitles_by_author_eng/president.txt
Archivo creado para 'paper': subtitles_by_author_eng/paper.txt
Archivo creado para 'book': subtitles_by_author_eng/book.txt
Archivo creado para 'sasaki': subtitles_by_author_eng/sasaki.txt
Archivo creado para 'c

In [115]:
import spacy
from sentence_transformers import SentenceTransformer, util

# Cargar modelos de spaCy para segmentar oraciones
nlp_en = spacy.load("en_core_web_sm")
nlp_es = spacy.load("es_core_news_sm")

def segment_text(text, nlp):
    """
    Segmenta el texto en oraciones usando spaCy.
    """
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    return sentences

def align_sentences(english_text, spanish_text):
    """
    Segmenta ambos textos en oraciones y alinea cada oración en inglés
    con la oración en español que más se le parezca semánticamente.
    
    Retorna una lista de tuplas (oración_en, oración_es, similitud).
    """
    # Segmentar los textos
    sentences_en = segment_text(english_text, nlp_en)
    sentences_es = segment_text(spanish_text, nlp_es)
    
    # Cargar un modelo multilingüe para obtener embeddings
    model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
    
    # Generar embeddings para ambas listas de oraciones
    embeddings_en = model.encode(sentences_en, convert_to_tensor=True)
    embeddings_es = model.encode(sentences_es, convert_to_tensor=True)
    
    # Calcular la matriz de similitud coseno
    cosine_scores = util.cos_sim(embeddings_en, embeddings_es)
    
    aligned_pairs = []
    # Usaremos un método greedy para emparejar: para cada oración en inglés,
    # elegimos la oración en español que tenga la mayor similitud y que aún no se haya usado.
    used_es = set()
    for i, en_sent in enumerate(sentences_en):
        scores = cosine_scores[i].cpu().numpy()
        # Ordena los índices de mayor a menor similitud
        sorted_indices = scores.argsort()[::-1]
        best_j = None
        for j in sorted_indices:
            if j not in used_es:
                best_j = j
                break
        if best_j is not None:
            aligned_pairs.append((en_sent, sentences_es[best_j], float(scores[best_j])))
            used_es.add(best_j)
        else:
            # Si todas las oraciones en español ya fueron usadas, se empareja sin restricción
            best_j = sorted_indices[0]
            aligned_pairs.append((en_sent, sentences_es[best_j], float(scores[best_j])))
    
    return aligned_pairs

# Ejemplo de textos completos
english_text = """ah, i knew it! the light feels best in the flesh!

a cursed spirit's flesh is so boring. where are the people? the women?! what a wonderful era to be in. women and children are crawling everywhere like maggots. marvelous! it will be a massacre!

how are you able to move?

he is suppressing me?"""

spanish_text = """¡lo sabía! ¡qué bien se siente la luz contra la piel!

la carne de un espectro no tiene gracia. ¿dónde está la gente? ¡¿y las mujeres?! pero qué buena época. mujeres y niños se arrastran por doquier como gusanos. ¡qué maravilla! ¡será una masacre!

¿cómo puedes moverte?

¿me está conteniendo?"""

# Alinear oraciones
aligned = align_sentences(english_text, spanish_text)

# Mostrar resultados
for en, es, score in aligned:
    test_f = open("output.txt", "w")
    test_f.write(f"""
    English: {en})
    Spanish: {es})
    Score: {score})\n\n
    """
    )
